In [2]:
pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 33.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torch]32m5/6 [torch]kx]
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install torch-geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [torch-geometric] [torch-geometric]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Ligand–Protein Catalytic Efficiency GNN (Prototype, Option 1)

import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem
from rdkit.Chem import rdmolops
import numpy as np

# ----------
# Utils: Molecule Graph
# ----------
def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    N = mol.GetNumAtoms()

    # Node features: atom number
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([atom.GetAtomicNum()])
    x = torch.tensor(atom_features, dtype=torch.float)

    # Edges: bonds
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

# ----------
# GNN Encoder (GCN)
# ----------
class GCNEncoder(nn.Module):
    def __init__(self, in_channels=1, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.relu(self.conv1(x, edge_index))
        x = self.relu(self.conv2(x, edge_index))
        return global_mean_pool(x, batch=None)  # Assume batch is None for single graph

# ----------
# MLP for Regression
# ----------
class RegressionHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

# ----------
# Full Model
# ----------
class LigandOnlyGNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = GCNEncoder()
        self.head = RegressionHead(32)

    def forward(self, mol_graph):
        emb = self.encoder(mol_graph)
        return self.head(emb)

# ----------
# Dummy Example
# ----------
example_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # Aspirin
mol_graph = mol_to_graph(example_smiles)

model = LigandOnlyGNNModel()
output = model(mol_graph)
print(f"Predicted catalytic efficiency (log10 scale): {output.item():.3f}")


In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Assuming you already have merged_df
merged_df["log10_efficiency"] = merged_df["log10kcat_max"] - merged_df["log10km_mean"]

# Select features and target
features = merged_df[["log10km_mean", "log10kcat_max", "value_kcat"]].values
target = merged_df["log10_efficiency"].values.reshape(-1, 1)

# Normalize features
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X = scaler_x.fit_transform(features)
y = scaler_y.fit_transform(target)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Define the model
model = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

# Loss and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(300):
    model.train()
    pred = model(X_train)
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_loss = loss_fn(test_pred, y_test)
    print(f"\nTest Loss: {test_loss.item():.4f}")
